In [1]:
import ollama
import requests
import json

In [2]:
model = "llama3.2:1b"

#### Task 1: Interact with deployed LLM via python 


**Objective:**

Explore different techniques to interact with the deployed LLM.

**Task Description:**

1. Use Request libaray (HTTP Client) and send a POST request to interact with the LLM: [How To](https://requests.readthedocs.io/en/latest/user/quickstart/#make-a-request)

In [3]:
# Simple HTTP Request via requests

# Define the URL of the deployed LLM ( this port is forwarded from the docker container to the host system)
url = "http://localhost:11434/api/generate"

# Define the prompt as json
body_json = {
    "model": model,
    "prompt": "Describe Generative AI in two sentences."
}

# ADD HERE YOUR CODE
# HINT: Send the POST request using the json body
response = requests.post(url, json=body_json)

# Check if the request was successful
if response.status_code == 200:
    # Process the response
    response_text = response.text

    # Convert each line to json
    response_lines = response_text.splitlines()
    response_json = [json.loads(line) for line in response_lines]
    for line in response_json:
        # Print the response. No line break
        print(line["response"], end="")
else:
    print("Error:", response.status_code)


Generative AI refers to a type of artificial intelligence that creates original content, such as text, images, or music, by generating new data based on patterns and relationships learned from existing data, rather than through explicit programming or instruction. This type of AI can produce highly varied and realistic outputs, including creative works that can be used for various purposes, such as artistic inspiration, entertainment, or educational purposes.

**Task Description:**

2. Use Ollama python library to interact with the LLM: [How To](https://pypi.org/project/ollama/)

- First use method ``ollama.chat(...)``
- First use method ``ollama.chat(...)`` with ``stream=True``

In [4]:
# API Call via ollama

# ADD HERE YOUR CODE

response = ollama.chat(
    model=model,
    messages=[{"role": "user", "content": "Describe Generative AI in two sentences."}],
)


print(response["message"]["content"])

Generative AI refers to a class of machine learning algorithms that can create original content, such as images, text, or music, by generating new examples that are similar to a given dataset, rather than simply reproducing existing data. These algorithms use complex neural networks and statistical models to learn patterns and relationships in the data, allowing them to produce novel and often surprising outputs that can be used for a wide range of applications, from art and design to education and entertainment.


In [5]:
# Streaming API Call via ollama

# Response streaming can be enabled by setting stream=True, 
# modifying function calls to return a Python generator where each part is an object in the stream.

# ADD HERE YOUR CODE

stream = ollama.chat(
    model=model,
    messages=[{"role": "user", "content": "Describe Generative AI in two sentences."}],
    stream=True,
)

for chunk in stream:
  print(chunk["message"]["content"], end="", flush=True)

Gener

ative

 AI

 refers

 to

 a

... (68 outputs left out)

 create

 original

 content

.

#### Task 2: Experimenting with Prompt Techniques

**Objective:**

Objective: Explore different prompt techniques (Zero Shot, One Shot, and Few Shot) by sending different types of prompts to the LLM.

![image](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*QSpK--jqPiUU_OHuZvtUWA.png)

**Task Description:**

1. Create three prompts for a sentiment analysis task: a Zero Shot prompt, a One Shot prompt, and a Few Shot prompt. Use the examples from the table above.
2. Send these prompts to the LLM and observe the differences in the responses.
3. Compare and discuss the responses.

In [6]:
# ADD HERE YOUR PROMPTS

zero_shot_prompt = """Classify the sentiment of the following text as positive, negative or neutral.
Answer with one word only.

Text: I think the vacation is okay.
Sentiment:"""

one_shot_prompt = """Classify the sentiment of the following text as positive, negative or neutral.
Answer with one word only.

Text: I love this movie!
Sentiment: positive

Text: I think the vacation is okay.
Sentiment:"""

few_shot_prompt = """Classify the sentiment of the following text as positive, negative or neutral.
Answer with one word only.

Text: I love this movie!
Sentiment: positive

Text: The food was cold and the service was terrible.
Sentiment: negative

Text: The train arrived at 9 o'clock.
Sentiment: neutral

Text: I think the vacation is okay.
Sentiment:"""

# Stream the responses and print them
# temperature=0 makes the answers reproducible, so the prompt techniques can be compared fairly.
# The same prompts are also sent to the bigger model llama3.2:3b (download once with: ollama pull llama3.2:3b)
for model_name in [model, "llama3.2:3b"]:
    print(f"\n==================== Model: {model_name} ====================")
    for idx, prompt in enumerate([zero_shot_prompt, one_shot_prompt, few_shot_prompt]):
        prompt_type = ["Zero-Shot", "One-Shot", "Few-Shot"][idx]
        print(f"\n--- {prompt_type} Prompt ---\n")
        print(f"User Prompt:\n{prompt}\n")

        stream = ollama.chat(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0},
            stream=True,
        )

        print("Model Output:")
        for chunk in stream:
            print(chunk["message"]["content"], end="", flush=True)
        print("\n-----------------------------\n")



==================== Model: llama3.2:1b ====================

--- Zero-Shot Prompt ---

User Prompt:
Classify the sentiment of the following text as positive, negative or neutral.
Answer with one word only.

Text: I think the vacation is okay.
Sentiment:

Model Output:


Negative


-----------------------------


--- One-Shot Prompt ---

User Prompt:
Classify the sentiment of the following text as positive, negative or neutral.
Answer with one word only.

Text: I love this movie!
Sentiment: positive

Text: I think the vacation is okay.
Sentiment:

Model Output:


Negative


-----------------------------


--- Few-Shot Prompt ---

User Prompt:
Classify the sentiment of the following text as positive, negative or neutral.
Answer with one word only.

Text: I love this movie!
Sentiment: positive

Text: The food was cold and the service was terrible.
Sentiment: negative

Text: The train arrived at 9 o'clock.
Sentiment: neutral

Text: I think the vacation is okay.
Sentiment:

Model Output:


Negative

... (3 outputs left out)

Neutral


-----------------------------


--- Few-Shot Prompt ---

User Prompt:
Classify the sentiment of the following text as positive, negative or neutral.
Answer with one word only.

Text: I love this movie!
Sentiment: positive

Text: The food was cold and the service was terrible.
Sentiment: negative

Text: The train arrived at 9 o'clock.
Sentiment: neutral

Text: I think the vacation is okay.
Sentiment:

Model Output:


neutral


-----------------------------



**Vergleich und Diskussion (Task 2):**

Erwartete Antwort für „I think the vacation is okay.“ ist **neutral**. Alle Prompts wurden mit `temperature=0` gesendet, damit die Ergebnisse reproduzierbar sind.

| Prompt | `llama3.2:1b` | `llama3.2:3b` |
|---|---|---|
| Zero-Shot | Negative ❌ | Neutral ✅ |
| One-Shot | Negative ❌ | Neutral ✅ |
| Few-Shot | Negative ❌ | neutral ✅ (kleingeschrieben wie in den Beispielen) |

- **Zero-Shot:** Das Modell bekommt nur die Aufgabe. Es muss selbst wissen, was „positive/negative/neutral“ bedeutet und wie die Antwort aussehen soll.
- **One-Shot:** Ein Beispiel zeigt das gewünschte Antwortformat (genau ein Label). Inhaltlich hilft ein einzelnes positives Beispiel kaum, weil das Modell daraus nicht lernt, wo die Grenze zu „neutral“ liegt.
- **Few-Shot:** Alle drei Klassen kommen als Beispiel vor. Das Modell übernimmt das Format sehr genau: `llama3.2:3b` antwortet hier kleingeschrieben, genau wie in den Beispielen.

**Fazit:** Prompt-Techniken verbessern vor allem das **Format** der Antwort. Ob die **Klassifikation** stimmt, hängt aber stark von der Modellgröße ab: Das sehr kleine `llama3.2:1b` stuft das grenzwertige „okay“ bei jeder Technik falsch als negativ ein, während `llama3.2:3b` bei allen drei Techniken richtig liegt. Ohne `temperature=0` schwanken die Antworten des 1B-Modells zusätzlich von Lauf zu Lauf (in früheren Läufen kamen „Positive“, „Negative“ und „Neutral“ vor).

#### Task 3: Prompt Refinement and Optimization

**Objective:** 

Refine a prompt to improve the clarity and quality of the LLM's response.

**Task Description:**

- Start with a basic prompt asking the LLM to summarize a paragraph.
- Refine the prompt by adding specific instructions to improve the summary's quality. (Example: define how long the summary should be, define on which to focus in the summary)

In [7]:
# Original prompt
original_prompt = "Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."

# ADD HERE YOUR PROMPT
refined_prompt = """Summarize the following paragraph in exactly one sentence of at most 25 words.
Focus on what Generative AI is and where it is used. Do not add any information that is not in the paragraph.
Return only the summary, without an introduction.

Paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."""

# Stream the responses and print them
for idx, prompt in enumerate([original_prompt, refined_prompt]):
    prompt_type = ["Original Prompt", "Refined Prompt"][idx]
    print(f"\n--- {prompt_type} ---\n")
    print(f"User Prompt:\n{prompt}\n")
    
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    print("Model Output:")
    for chunk in stream:
        print(chunk["message"]["content"], end="", flush=True)
    print("\n-----------------------------\n")



--- Original Prompt ---

User Prompt:
Summarize the following paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries.

Model Output:


Gener

ative

 AI

 is

 a

... (64 outputs left out)

 creative

 industries

.


-----------------------------



#### [Optional] Task 4: Structured Prompting with Roles (Pirate Theme)

**Objective:**

Learn how to use structured prompts that combine role assignment, clear instructions, and examples to improve the output of language models. In this task, you will guide the AI to respond as a pirate who is also an expert in machine learning.

**Instructions:**

- Role Assignment: In your prompt, specify the role of the AI as a Machine Learning Expert who speaks like a pirate.

- Instruction: Clearly state what you want the AI to explain or discuss in pirate language.

- Examples: Provide examples to guide the AI in using pirate lingo while explaining technical concepts.

In [8]:
# Combined Techniques Prompt with Pirate Theme

structured_prompt = """
Role: You are a Machine Learning Expert who always speaks like a pirate.

Instruction: Explain to a beginner what overfitting is and how to prevent it.
Use pirate language (e.g. "Arr", "matey", "ye", "treasure", "the seven seas"), but keep the technical content correct.
Keep the answer under 150 words.

Examples:
- Question: What is training data?
  Answer: Arr matey, training data be the treasure map me model studies before settin' sail!
- Question: What is a neural network?
  Answer: Ahoy! A neural network be a crew o' tiny sailors, each passin' messages to the next 'til the captain makes a decision!

Question: What is overfitting and how can ye prevent it?
Answer:
"""

# Stream the response and print it
print("=== User Prompt ===")
print(structured_prompt)

stream = ollama.chat(
    model=model,
    messages=[{"role": "user", "content": structured_prompt}],
    stream=True,
)

print("\n=== Model Output ===")
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print("\n")


=== User Prompt ===

Role: You are a Machine Learning Expert who always speaks like a pirate.

Instruction: Explain to a beginner what overfitting is and how to prevent it.
Use pirate language (e.g. "Arr", "matey", "ye", "treasure", "the seven seas"), but keep the technical content correct.
Keep the answer under 150 words.

Examples:
- Question: What is training data?
  Answer: Arr matey, training data be the treasure map me model studies before settin' sail!
- Question: What is a neural network?
  Answer: Ahoy! A neural network be a crew o' tiny sailors, each passin' messages to the next 'til the captain makes a decision!

Question: What is overfitting and how can ye prevent it?
Answer:


=== Model Output ===


Ye

 be

 wonder

in

'

... (194 outputs left out)

',

 me

 hearty

!